# B-Roll Editor
**Three steps:**
1. Run Cell 1 — installs everything (2 min, once per session)
2. Run Cell 2 — paste your free Gemini API key
3. Run Cell 3 — upload your video, wait, download the result

To run a cell: tap it, then tap the play button ▶ on the left

In [ ]:
# CELL 1: Install everything (run once per session, takes ~2 min)
import subprocess, sys
print('Installing ffmpeg...')
subprocess.run(['apt-get', 'install', '-y', 'ffmpeg'], capture_output=True)
print('Installing Python packages...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'google-generativeai', 'faster-whisper', 'requests'], check=True)
print('Done! Run the next cell.')

In [ ]:
# CELL 2: Paste your FREE Gemini API key
#
# Get it free (no credit card) in 30 seconds:
#   1. Go to aistudio.google.com
#   2. Sign in with your Google account
#   3. Click 'Get API key' → 'Create API key'
#   4. Copy it and paste below

GEMINI_API_KEY = "paste-your-key-here"

if GEMINI_API_KEY == "paste-your-key-here":
    print('Paste your key above where it says paste-your-key-here, then re-run this cell.')
else:
    print('API key set! Run the next cell.')

In [ ]:
# CELL 3: Upload video → process → download
import os, json, subprocess, requests, uuid, shutil
from google.colab import files

PEXELS_API_KEY = "LppRQMMFSN0E7avfYQUoeQVdifahsxZwB0Uzag6Z7OQhZvemwAYfQ5eu"

print('Choose your MP4 file...')
uploaded = files.upload()
video_filename = list(uploaded.keys())[0]
print(f'Uploaded: {video_filename}')

def run_ff(*args, label='ffmpeg'):
    r = subprocess.run(['ffmpeg','-y']+list(args), capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f'{label} failed:\n{r.stderr[-1500:]}')

def get_info(path):
    r = subprocess.run(['ffprobe','-v','quiet','-print_format','json',
                        '-show_streams','-show_format', path],
                       capture_output=True, text=True, check=True)
    d = json.loads(r.stdout)
    w = h = None
    for s in d.get('streams',[]):
        if s.get('codec_type') == 'video':
            w, h = s['width'], s['height']; break
    return w, h, float(d.get('format',{}).get('duration', 0))

def transcribe(audio_path, duration):
    from faster_whisper import WhisperModel
    print('  Loading Whisper model (~150 MB on first run)...')
    model = WhisperModel('base', device='cpu', compute_type='int8')
    segs, _ = model.transcribe(audio_path, beam_size=5,
                                word_timestamps=True, vad_filter=True)
    transcript, chunk_start, chunk_words, chunk_end = [], None, [], 0.0
    for seg in segs:
        for w in (seg.words or []):
            if chunk_start is None: chunk_start = w.start
            chunk_words.append(w.word); chunk_end = w.end
            dur = chunk_end - chunk_start
            if w.word.strip().endswith(('.','!','?',',')) and dur >= 3.0 or dur >= 15.0:
                transcript.append({'start': round(chunk_start,2), 'end': round(chunk_end,2),
                                   'text': ''.join(chunk_words).strip()})
                chunk_start, chunk_words = None, []
    if chunk_words and chunk_start is not None:
        transcript.append({'start': round(chunk_start,2), 'end': round(duration,2),
                           'text': ''.join(chunk_words).strip()})
    elif transcript:
        transcript[-1]['end'] = round(duration, 2)
    return transcript or [{'start': 0.0, 'end': duration, 'text': '(no speech)'}]

def plan_edit(transcript, duration):
    import google.generativeai as genai
    genai.configure(api_key=GEMINI_API_KEY)
    model = genai.GenerativeModel('gemini-1.5-flash')
    lines = '\n'.join(f"[{s['start']:.1f}s-{s['end']:.1f}s]: {s['text']}" for s in transcript)
    prompt = (
        'You are a video editor. Analyze this transcript and plan the edit.\n\n'
        f'TRANSCRIPT:\n{lines}\n\n'
        'RULES:\n'
        '1. face_cam: hooks (opening), CTAs, emotional moments, direct address to viewer\n'
        '2. broll: describing products, places, concepts, tips, steps — anything visual\n'
        '3. Minimum 3 seconds per segment\n'
        '4. Aim for 40-60% b-roll\n'
        f'5. Segments must cover 0.0 to {duration:.1f}s with no gaps\n'
        '6. For broll: a 2-4 word Pexels search keyword (concrete, visual)\n'
        '7. First segment is almost always face_cam\n\n'
        'Return ONLY a JSON array, no markdown, no explanation:\n'
        '[\n'
        '  {"start": 0.0, "end": 9.0, "type": "face_cam"},\n'
        '  {"start": 9.0, "end": 18.0, "type": "broll", "keyword": "coffee morning desk"},\n'
        f'  {{"start": 18.0, "end": {duration:.1f}, "type": "face_cam"}}\n'
        ']\n'
        f'The last segment MUST end at exactly {duration:.1f}'
    )
    resp = model.generate_content(prompt)
    text = resp.text.strip()
    if text.startswith('```'):
        parts = text.split('```'); text = parts[1]
        if text.startswith('json'): text = text[4:]
    segs = json.loads(text.strip())
    if segs: segs[-1]['end'] = duration
    merged = [segs[0]]
    for s in segs[1:]:
        if s['end'] - s['start'] < 3.0: merged[-1]['end'] = s['end']
        else: merged.append(s)
    return merged

def get_pexels(kw):
    r = requests.get('https://api.pexels.com/videos/search',
                     headers={'Authorization': PEXELS_API_KEY},
                     params={'query': kw, 'orientation': 'portrait', 'per_page': 10}, timeout=30)
    for vid in r.json().get('videos', []):
        for vf in sorted(vid.get('video_files', []), key=lambda x: x.get('height',0), reverse=True):
            if vf.get('width', 9999) < vf.get('height', 0): return vf['link']
    vids = r.json().get('videos', [])
    return vids[0]['video_files'][0]['link'] if vids and vids[0].get('video_files') else None

# Pipeline
work = f'job_{uuid.uuid4().hex[:8]}'; os.makedirs(work)

print('\n[1/7] Reading video info...')
w, h, duration = get_info(video_filename)
print(f'      {w}x{h}, {duration:.1f}s')

print('[2/7] Extracting audio...')
audio = f'{work}/audio.mp3'
run_ff('-i', video_filename, '-vn', '-acodec', 'libmp3lame',
       '-ar', '16000', '-ac', '1', '-b:a', '32k', audio)

print('[3/7] Transcribing audio (Whisper)...')
transcript = transcribe(audio, duration)
print(f'      {len(transcript)} segments')

print('[4/7] Planning edit with Gemini...')
segments = plan_edit(transcript, duration)
broll_pct = sum(s['end']-s['start'] for s in segments if s['type']=='broll') / duration * 100
print(f'      {len(segments)} segments, {broll_pct:.0f}% b-roll')
for s in segments:
    kw = f" [{s['keyword']}]" if s.get('keyword') else ''
    print(f"      {s['start']:.1f}s -> {s['end']:.1f}s  {s['type']}{kw}")

print('[5/7] Downloading b-roll from Pexels...')
broll_clips = {}
for i, seg in enumerate([s for s in segments if s['type']=='broll']):
    kw = seg.get('keyword', 'nature landscape')
    print(f'      "{kw}"...')
    url = get_pexels(kw)
    if url:
        p = f'{work}/broll_{i}.mp4'
        r = requests.get(url, stream=True, timeout=120)
        with open(p, 'wb') as f:
            for chunk in r.iter_content(65536): f.write(chunk)
        broll_clips[i] = p; print('      downloaded')
    else:
        broll_clips[i] = None; print('      not found, using face cam')

print('[6/7] Cutting and compositing...')
seg_files = []; broll_idx = 0
for i, seg in enumerate(segments):
    dur = seg['end'] - seg['start']; out = f'{work}/seg_{i:03d}.mp4'
    if seg['type'] == 'face_cam':
        run_ff('-ss', str(seg['start']), '-i', video_filename, '-t', str(dur),
               '-vf', f'scale={w}:{h}:force_original_aspect_ratio=decrease,pad={w}:{h}:(ow-iw)/2:(oh-ih)/2:black',
               '-an', '-c:v', 'libx264', '-preset', 'fast', '-crf', '22', out)
    else:
        clip = broll_clips.get(broll_idx); broll_idx += 1
        if clip:
            run_ff('-stream_loop', '-1', '-i', clip, '-t', str(dur),
                   '-vf', f'scale={w}:{h}:force_original_aspect_ratio=increase,crop={w}:{h}',
                   '-an', '-c:v', 'libx264', '-preset', 'fast', '-crf', '22', out)
        else:
            run_ff('-ss', str(seg['start']), '-i', video_filename, '-t', str(dur),
                   '-vf', f'scale={w}:{h}:force_original_aspect_ratio=decrease,pad={w}:{h}:(ow-iw)/2:(oh-ih)/2:black',
                   '-an', '-c:v', 'libx264', '-preset', 'fast', '-crf', '22', out)
    seg_files.append(out)

concat = f'{work}/concat.txt'
with open(concat, 'w') as f:
    for sp in seg_files: f.write(f"file '{os.path.abspath(sp)}'\n")
video_only = f'{work}/video_only.mp4'
run_ff('-f', 'concat', '-safe', '0', '-i', concat, '-c:v', 'copy', video_only)
output = f'{work}/output.mp4'
run_ff('-i', video_only, '-i', video_filename,
       '-map', '0:v:0', '-map', '1:a:0',
       '-c:v', 'copy', '-c:a', 'aac', '-b:a', '192k', '-shortest', output)

print(f'[7/7] Done! {os.path.getsize(output)/1024/1024:.1f} MB')
print('Downloading your finished video...')
files.download(output)
shutil.rmtree(work)